In [13]:
# =====================================
# ASL CNN PREDICTION / TESTING
# =====================================

import os
import cv2
import numpy as np
import tensorflow as tf

# -------- PATHS --------
BASE_DIR = r"C:\Users\Dell\Desktop\asl-sign-recognition"
MODEL_DIR = os.path.join(BASE_DIR, "models")
# -----------------------

# Load trained model
model = tf.keras.models.load_model(
    os.path.join(MODEL_DIR, "asl_model_full.keras")
)

# Load labels
labels = np.load(
    os.path.join(MODEL_DIR, "labels.npy"),
    allow_pickle=True
)

print("✅ Model & labels loaded")
print("Classes:", labels)

# -------- SETTINGS --------
IMG_SIZE = 64
# -------------------------

def preprocess_image(img_path):
    """Same preprocessing used during TRAINING (advanced)"""
    img = cv2.imread(img_path)

    if img is None:
        raise ValueError("❌ Image not found")

    # Resize
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # CLAHE (contrast normalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # Gaussian blur
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    # Normalize
    gray = gray.astype("float32") / 255.0

    # Add channel + batch dimension → (1, 64, 64, 1)
    gray = np.expand_dims(gray, axis=-1)
    gray = np.expand_dims(gray, axis=0)

    return gray



def predict_image(img_path, confidence_threshold=0.5):
    img = preprocess_image(img_path)

    predictions = model.predict(img, verbose=0)[0]
    idx = np.argmax(predictions)
    confidence = float(predictions[idx])

    if confidence < confidence_threshold:
        return "UNKNOWN", confidence

    return labels[idx], confidence


# =============================
# TEST IMAGE (CHANGE THIS PATH)
# =============================
TEST_IMAGE = r"C:\Users\Dell\Desktop\python\Test_Images\test_hand.jpg"
label, conf = predict_image(TEST_IMAGE)

print("\n🔮 Prediction Result")
print("Predicted Label:", label)
print(f"Confidence: {conf * 100:.2f}%")


✅ Model & labels loaded
Classes: ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'R' 'Sorry' 'hello' 'please']

🔮 Prediction Result
Predicted Label: I
Confidence: 52.33%
